# Tarea 2: Aprendizaje Distribuido para Computer Vision
## LEAD University — 2026

| Campo | Valor |
|---|---|
| **Estudiante** | Jason Jesús Barrantes Sánchez |
| **Profesor** | Johansell Villalobos Cubillo |
| **Fecha** | 2026-06-03 |
| **Dataset** | Wonders of the World Image Classification (Kaggle) |
| **Frameworks** | JAX · Flax NNX · Optax |

---
Este notebook implementa un clasificador de imágenes de maravillas del mundo usando **JAX** y **Flax NNX**.  
Se exploran: tamaño de lote, tasa de aprendizaje, tamaño de red, optimización de hiperparámetros y precisión numérica.

## Sección 1: Configuración del Entorno

In [ ]:
# ── 1.1 Instalación de dependencias ─────────────────────────────────────────
# En Colab, JAX suele estar preinstalado; sólo necesitamos flax y optax actualizados.
!pip install -q --upgrade flax optax kaggle tqdm matplotlib seaborn pandas Pillow

import importlib, subprocess, sys

def check_install(package):
    try:
        importlib.import_module(package)
        print(f"  ✓ {package}")
    except ImportError:
        print(f"  ✗ {package} NO encontrado")

for pkg in ["jax", "flax", "optax", "PIL", "matplotlib", "seaborn", "pandas", "tqdm"]:
    check_install(pkg)

In [ ]:
# ── 1.2 Importaciones ───────────────────────────────────────────────────────
import os, time, gc, warnings, random
import json
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
import seaborn as sns
from PIL import Image
from tqdm.notebook import tqdm

import jax
import jax.numpy as jnp
from jax import lax
from flax import nnx
import optax

warnings.filterwarnings("ignore")
plt.rcParams.update({"figure.facecolor":"white","axes.facecolor":"#f8f9fa",
                     "font.size":11,"axes.titlesize":12})
sns.set_palette("husl")

# Reproducibilidad
SEED = 42
np.random.seed(SEED)
random.seed(SEED)

print("✓ Librerías importadas")
print(f"  JAX   : {jax.__version__}")
print(f"  Flax  : {__import__('flax').__version__}")
print(f"  Optax : {optax.__version__}")

In [ ]:
# ── 1.3 Verificación del Acelerador ─────────────────────────────────────────
print("=" * 55)
print("  DISPOSITIVOS JAX")
print("=" * 55)
devices = jax.devices()
print(f"  Dispositivos : {devices}")
print(f"  Cantidad     : {jax.device_count()}")
print(f"  Backend      : {jax.default_backend()}")

if jax.default_backend() in ("gpu","tpu"):
    print(f"\n  ✓ Acelerador activo: {jax.default_backend().upper()}")
    for i,d in enumerate(devices):
        print(f"    [{i}] {d}")
else:
    print("\n  ⚠ Sólo CPU detectada.")
    print("  Activar GPU: Entorno de ejecución → Cambiar tipo → GPU/TPU")

# Captura de pantalla recomendada aquí ↑

# Prueba rápida
x = jnp.ones((4,4))
y = jnp.dot(x,x)
print(f"\n  ✓ Prueba JAX: dot(ones(4,4), ones(4,4)) = {y[0,0]:.0f}")

# ── 2.1 Descarga del Dataset desde Kaggle ───────────────────────────────────
# El nuevo formato de token de Kaggle (KGAT_...) se configura via
# variable de entorno — no necesita kaggle.json.
#
# Pasos:
#   1. Ve a kaggle.com → Settings → API → Create New Token  (nuevo KGAT_...)
#   2. Pega el token en la línea de abajo entre las comillas
#   3. Ejecuta esta celda

import os
from pathlib import Path

# ── PEGA TU TOKEN AQUÍ ────────────────────────────────────────────────────────
KAGGLE_TOKEN = "KGAT_XXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXX"   # <-- reemplazar
# ─────────────────────────────────────────────────────────────────────────────

assert KAGGLE_TOKEN != "KGAT_XXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXX", \
    "❌ Reemplaza KAGGLE_TOKEN con tu token real antes de ejecutar."

os.environ["KAGGLE_API_TOKEN"] = KAGGLE_TOKEN

# Descargar y extraer
DATA_DIR = Path("./data")
DATA_DIR.mkdir(exist_ok=True)

!pip install -q kaggle
!kaggle datasets download -d balabaskar/wonders-of-the-world-image-classification \
       -p ./data --unzip

# Verificar descarga
print("\n📁 Contenido de ./data:")
for item in sorted(DATA_DIR.iterdir()):
    print(f"  {item.name}/")


In [ ]:
# ── 2.1 Descarga del Dataset desde Kaggle ───────────────────────────────────
import os
from google.colab import files

# Configurar credenciales
os.makedirs("/root/.kaggle", exist_ok=True)

print("Suba su archivo kaggle.json:")
uploaded = files.upload()

for fn in uploaded:
    if "kaggle" in fn.lower():
        os.rename(fn, "/root/.kaggle/kaggle.json")
        os.chmod("/root/.kaggle/kaggle.json", 0o600)
        print("✓ Credenciales configuradas")
        break

# Descargar y extraer
DATA_DIR = Path("./data")
DATA_DIR.mkdir(exist_ok=True)

!kaggle datasets download -d balabaskar/wonders-of-the-world-image-classification -p ./data --unzip

# Listar contenido
print("\n📁 Contenido de ./data:")
for item in sorted(DATA_DIR.iterdir()):
    print(f"  {item.name}/")

In [ ]:
# ── 2.2 Detección automática del directorio raíz ────────────────────────────
def find_dataset_root(base: Path) -> Path:
    """Busca el directorio que contiene subcarpetas con imágenes."""
    EXTENSIONS = {".jpg",".jpeg",".png",".JPG",".JPEG",".PNG"}
    for root, dirs, files in os.walk(base):
        rp = Path(root)
        for d in dirs:
            imgs = [f for f in (rp/d).iterdir()
                    if f.is_file() and f.suffix in EXTENSIONS]
            if len(imgs) > 0:
                return rp
    return base

DATASET_ROOT = find_dataset_root(DATA_DIR)
print(f"✓ Raíz del dataset: {DATASET_ROOT}")

# Analizar clases
EXTENSIONS = {".jpg",".jpeg",".png",".JPG",".JPEG",".PNG"}
classes = sorted([d.name for d in DATASET_ROOT.iterdir() if d.is_dir()])
class_counts = {}
total = 0
print(f"\n📊 Clases encontradas: {len(classes)}")
print("-" * 35)
for cls in classes:
    n = len([f for f in (DATASET_ROOT/cls).iterdir()
             if f.is_file() and f.suffix in EXTENSIONS])
    class_counts[cls] = n
    total += n
    print(f"  {cls:<25}: {n:>5} imágenes")
print(f"  {'TOTAL':<25}: {total:>5} imágenes")
NUM_CLASSES = len(classes)

In [ ]:
# ── 2.3 Visualización de muestras y distribución ────────────────────────────
fig = plt.figure(figsize=(16, 10))
n_show = min(len(classes), 10)
gs = gridspec.GridSpec(2, 5, figure=fig)

for i, cls in enumerate(classes[:n_show]):
    cls_dir = DATASET_ROOT / cls
    img_paths = [f for f in cls_dir.iterdir()
                 if f.suffix.lower() in {".jpg",".jpeg",".png"}]
    if not img_paths:
        continue
    ax = fig.add_subplot(gs[i//5, i%5])
    img = Image.open(img_paths[0]).convert("RGB").resize((128,128))
    ax.imshow(img)
    ax.set_title(cls[:20], fontsize=8, fontweight="bold")
    ax.axis("off")

plt.suptitle("Muestras del Dataset — Wonders of the World", fontsize=13, fontweight="bold")
plt.tight_layout()
plt.savefig("dataset_samples.png", dpi=150, bbox_inches="tight")
plt.show()

# Distribución de clases
fig, ax = plt.subplots(figsize=(12, 4))
bars = ax.bar(class_counts.keys(), class_counts.values(),
              color=sns.color_palette("husl", len(class_counts)),
              edgecolor="black", linewidth=0.5)
ax.set_title("Distribución de Imágenes por Clase", fontweight="bold")
ax.set_ylabel("Número de Imágenes")
ax.tick_params(axis="x", rotation=45)
ax.grid(True, alpha=0.3, axis="y")
plt.tight_layout()
plt.savefig("class_distribution.png", dpi=150, bbox_inches="tight")
plt.show()

## Sección 3: Preprocesamiento y Pipeline de Datos

In [ ]:
# ── 3.1 Clase WondersDataset ────────────────────────────────────────────────
IMG_SIZE   = 64   # Resolución de las imágenes: 64×64 px
CHANNELS   = 3    # RGB
EXTENSIONS = {".jpg",".jpeg",".png",".JPG",".JPEG",".PNG"}

class WondersDataset:
    """
    Pipeline de datos para Wonders of the World.
    - Divide en train/val/test
    - Normaliza con media y desv. estándar del conjunto de entrenamiento
    - Soporta data augmentation básica
    """

    def __init__(self, root_dir, img_size=64,
                 val_split=0.15, test_split=0.15, seed=42):
        self.root_dir = Path(root_dir)
        self.img_size = img_size

        # ── Clases
        self.classes = sorted([d.name for d in self.root_dir.iterdir() if d.is_dir()])
        self.class_to_idx = {c: i for i, c in enumerate(self.classes)}
        self.num_classes  = len(self.classes)

        # ── Rutas y etiquetas
        all_paths, all_labels = [], []
        for cls in self.classes:
            for f in (self.root_dir/cls).iterdir():
                if f.is_file() and f.suffix in EXTENSIONS:
                    all_paths.append(str(f))
                    all_labels.append(self.class_to_idx[cls])

        # Mezclar
        rng = np.random.RandomState(seed)
        idx = rng.permutation(len(all_paths))
        all_paths  = [all_paths[i]  for i in idx]
        all_labels = [all_labels[i] for i in idx]

        # ── Split
        n       = len(all_paths)
        n_val   = int(n * val_split)
        n_test  = int(n * test_split)
        n_train = n - n_val - n_test

        self.train_paths  = all_paths[:n_train]
        self.train_labels = all_labels[:n_train]
        self.val_paths    = all_paths[n_train:n_train+n_val]
        self.val_labels   = all_labels[n_train:n_train+n_val]
        self.test_paths   = all_paths[n_train+n_val:]
        self.test_labels  = all_labels[n_train+n_val:]

        print(f"✓ Dataset cargado — {self.num_classes} clases")
        print(f"  Train : {len(self.train_paths):4d} | "
              f"Val : {len(self.val_paths):4d} | "
              f"Test: {len(self.test_paths):4d}")

        self._compute_stats()

    # ── Normalización ────────────────────────────────────────────────────
    def _compute_stats(self, n_samples: int = 500):
        paths = self.train_paths[:min(n_samples, len(self.train_paths))]
        imgs  = []
        for p in paths:
            try:
                img = Image.open(p).convert("RGB").resize(
                    (self.img_size, self.img_size), Image.BILINEAR)
                imgs.append(np.array(img, dtype=np.float32) / 255.0)
            except Exception:
                continue
        imgs = np.stack(imgs)
        self.mean = imgs.mean(axis=(0,1,2))
        self.std  = imgs.std(axis=(0,1,2)) + 1e-8
        print(f"  Media RGB: {self.mean.round(3)} | Std RGB: {self.std.round(3)}")

    # ── Carga de una imagen ──────────────────────────────────────────────
    def load_image(self, path: str, augment: bool = False) -> np.ndarray:
        img = Image.open(path).convert("RGB").resize(
            (self.img_size, self.img_size), Image.BILINEAR)
        img = np.array(img, dtype=np.float32) / 255.0

        if augment:
            if np.random.random() > 0.5:          # Flip horizontal
                img = img[:, ::-1, :]
            if np.random.random() > 0.5:           # Flip vertical (suave)
                img = np.clip(img * np.random.uniform(0.8, 1.2), 0, 1)

        return ((img - self.mean) / self.std).astype(np.float32)

    # ── Generador de batches ─────────────────────────────────────────────
    def get_batches(self, split: str = "train", batch_size: int = 32,
                    shuffle: bool = True, augment: bool = False):
        paths_map  = {"train": self.train_paths,
                      "val":   self.val_paths,
                      "test":  self.test_paths}
        labels_map = {"train": self.train_labels,
                      "val":   self.val_labels,
                      "test":  self.test_labels}
        paths, labels = paths_map[split][:], labels_map[split][:]

        if shuffle:
            perm   = np.random.permutation(len(paths))
            paths  = [paths[i]  for i in perm]
            labels = [labels[i] for i in perm]

        for start in range(0, len(paths), batch_size):
            bp = paths[start:start+batch_size]
            bl = labels[start:start+batch_size]
            imgs, lbs = [], []
            for p, l in zip(bp, bl):
                try:
                    imgs.append(self.load_image(p, augment=augment))
                    lbs.append(l)
                except Exception:
                    continue
            if imgs:
                yield {"image": np.array(imgs, dtype=np.float32),
                       "label": np.array(lbs,  dtype=np.int32)}

# ── Inicializar dataset
dataset = WondersDataset(DATASET_ROOT, img_size=IMG_SIZE)
NUM_CLASSES = dataset.num_classes

# Verificar un batch
sample_batch = next(dataset.get_batches("train", batch_size=8, shuffle=False))
print(f"\n✓ Batch de prueba — imágenes: {sample_batch['image'].shape}, "
      f"etiquetas: {sample_batch['label']}")

## Sección 4: Modelo CNN con Flax NNX

### Arquitectura implementada

```
Input (batch, 64, 64, 3)
  ↓  Conv(3→F, 3×3, SAME) → BatchNorm → ReLU → MaxPool(2×2)
  ↓  Conv(F→2F, 3×3, SAME) → BatchNorm → ReLU → MaxPool(2×2)
  ↓  Conv(2F→4F, 3×3, SAME) → BatchNorm → ReLU → GlobalAvgPool
  ↓  Dense(4F→D) → ReLU → Dropout(0.3)
  ↓  Dense(D→C) → Logits
```

donde **F** = num_filters, **D** = dense_size, **C** = num_classes

In [ ]:
# ── 4.1 Definición de la CNN con Flax NNX ───────────────────────────────────

def max_pool2d(x: jnp.ndarray,
               window: tuple = (2, 2),
               strides: tuple = (2, 2),
               padding: str = "VALID") -> jnp.ndarray:
    """Max pooling 2D usando jax.lax (compatible con todas las versiones de Flax)."""
    return lax.reduce_window(
        x,
        init_value=-jnp.inf,
        computation=lax.max,
        window_dimensions=(1,) + window + (1,),
        window_strides=(1,) + strides + (1,),
        padding=padding,
    )


class CNN(nnx.Module):
    """
    Red Neuronal Convolucional implementada con Flax NNX.

    Parámetros
    ----------
    num_classes  : int   — número de clases de salida
    num_filters  : int   — filtros base (bloques usan F, 2F, 4F)
    dense_size   : int   — neuronas en la capa densa oculta
    dropout_rate : float — tasa de dropout para regularización
    rngs         : nnx.Rngs
    """

    def __init__(self,
                 num_classes:  int,
                 num_filters:  int = 64,
                 dense_size:   int = 256,
                 dropout_rate: float = 0.3,
                 rngs: nnx.Rngs = None):

        F = num_filters

        # ── Bloque 1
        self.conv1 = nnx.Conv(3,   F,   kernel_size=(3,3), strides=(1,1),
                              padding="SAME", rngs=rngs)
        self.bn1   = nnx.BatchNorm(F,   rngs=rngs)

        # ── Bloque 2
        self.conv2 = nnx.Conv(F,   F*2, kernel_size=(3,3), strides=(1,1),
                              padding="SAME", rngs=rngs)
        self.bn2   = nnx.BatchNorm(F*2, rngs=rngs)

        # ── Bloque 3
        self.conv3 = nnx.Conv(F*2, F*4, kernel_size=(3,3), strides=(1,1),
                              padding="SAME", rngs=rngs)
        self.bn3   = nnx.BatchNorm(F*4, rngs=rngs)

        # ── Clasificador
        self.dense1    = nnx.Linear(F*4, dense_size, rngs=rngs)
        self.dense_out = nnx.Linear(dense_size, num_classes, rngs=rngs)
        self.dropout   = nnx.Dropout(rate=dropout_rate, rngs=rngs)

    def __call__(self, x: jnp.ndarray,
                 use_running_average: bool = False) -> jnp.ndarray:
        """
        Forward pass.

        Args
        ----
        x                   : (batch, H, W, C)
        use_running_average : False → entrenamiento | True → evaluación
        """
        # Bloque 1 — Conv → BN → ReLU → MaxPool
        x = self.conv1(x)
        x = self.bn1(x, use_running_average=use_running_average)
        x = jax.nn.relu(x)
        x = max_pool2d(x)

        # Bloque 2 — Conv → BN → ReLU → MaxPool
        x = self.conv2(x)
        x = self.bn2(x, use_running_average=use_running_average)
        x = jax.nn.relu(x)
        x = max_pool2d(x)

        # Bloque 3 — Conv → BN → ReLU → GlobalAvgPool
        x = self.conv3(x)
        x = self.bn3(x, use_running_average=use_running_average)
        x = jax.nn.relu(x)
        x = x.mean(axis=(1, 2))          # (batch, 4F)

        # Clasificador
        x = self.dense1(x)
        x = jax.nn.relu(x)
        x = self.dropout(x)
        x = self.dense_out(x)
        return x


# ── Verificar arquitectura con un forward pass ───────────────────────────────
_test = CNN(num_classes=NUM_CLASSES, num_filters=32, dense_size=128,
            rngs=nnx.Rngs(SEED))
_x    = jnp.ones((4, IMG_SIZE, IMG_SIZE, 3))
_y    = _test(_x, use_running_average=True)
print(f"✓ Forward pass: {_x.shape}  →  {_y.shape}")
del _test

print(f"\nArquitectura (F=filtros, D=dense, C=clases):")
print(f"  Input   : (batch, {IMG_SIZE}, {IMG_SIZE}, 3)")
print(f"  Block 1 : Conv(3→F)   → BN → ReLU → MaxPool(2×2)")
print(f"  Block 2 : Conv(F→2F)  → BN → ReLU → MaxPool(2×2)")
print(f"  Block 3 : Conv(2F→4F) → BN → ReLU → GlobalAvgPool")
print(f"  Dense   : Linear(4F→D) → ReLU → Dropout")
print(f"  Output  : Linear(D→{NUM_CLASSES}) → Logits")

## Sección 5: Funciones de Entrenamiento

In [ ]:
# ── 5.1 Paso de entrenamiento (con diferenciación automática + JIT) ──────────

@nnx.jit
def train_step(model: CNN, optimizer: nnx.Optimizer,
               images: jnp.ndarray, labels: jnp.ndarray):
    """
    Paso de entrenamiento único.
    Usa diferenciación automática (nnx.value_and_grad) y compilación JIT.
    """
    def loss_fn(model):
        logits = model(images, use_running_average=False)
        loss   = optax.softmax_cross_entropy_with_integer_labels(
                     logits, labels).mean()
        return loss, logits

    # Gradientes mediante diferenciación automática
    grad_fn = nnx.value_and_grad(loss_fn, has_aux=True)
    (loss, logits), grads = grad_fn(model)

    # Actualizar parámetros (descenso de gradiente via Adam)
    optimizer.update(grads)

    acc = jnp.mean(jnp.argmax(logits, axis=-1) == labels)
    return loss, acc


@nnx.jit
def eval_step(model: CNN,
              images: jnp.ndarray, labels: jnp.ndarray):
    """Paso de evaluación (sin actualización de parámetros)."""
    logits = model(images, use_running_average=True)
    loss   = optax.softmax_cross_entropy_with_integer_labels(
                 logits, labels).mean()
    acc    = jnp.mean(jnp.argmax(logits, axis=-1) == labels)
    return loss, acc


# ── 5.2 Epoch completa ───────────────────────────────────────────────────────
def run_epoch(model, optimizer, dataset, batch_size,
              split="train", augment=False):
    losses, accs = [], []
    training = (split == "train")
    for batch in dataset.get_batches(split=split, batch_size=batch_size,
                                     shuffle=training, augment=augment):
        imgs = jnp.array(batch["image"])
        lbs  = jnp.array(batch["label"])
        if training:
            loss, acc = train_step(model, optimizer, imgs, lbs)
        else:
            loss, acc = eval_step(model, imgs, lbs)
        losses.append(float(loss))
        accs.append(float(acc))
    return (np.mean(losses) if losses else float("nan"),
            np.mean(accs)   if accs   else 0.0)


# ── 5.3 Función principal de entrenamiento ───────────────────────────────────
def train_and_evaluate(config: dict, dataset: WondersDataset,
                       verbose: bool = True) -> dict:
    """
    Entrenamiento + evaluación completos.

    Parámetros (config)
    -------------------
    num_epochs, batch_size, learning_rate,
    num_filters, dense_size, augment
    """
    num_epochs    = config.get("num_epochs",    20)
    batch_size    = config.get("batch_size",    64)
    learning_rate = config.get("learning_rate", 1e-3)
    num_filters   = config.get("num_filters",   64)
    dense_size    = config.get("dense_size",    256)
    augment       = config.get("augment",       True)

    if verbose:
        print(f"\n{'='*65}")
        print(f"  lr={learning_rate:.0e}  batch={batch_size}  "
              f"filtros={num_filters}  dense={dense_size}  épocas={num_epochs}")
        print(f"{'='*65}")

    # Crear modelo y optimizador
    model     = CNN(num_classes=dataset.num_classes,
                    num_filters=num_filters,
                    dense_size=dense_size,
                    rngs=nnx.Rngs(SEED))
    optimizer = nnx.Optimizer(model, optax.adam(learning_rate))

    history = {"train_loss":[],"train_acc":[],
               "val_loss":[],"val_acc":[],"epoch_times":[]}

    t0 = time.time()
    log_every = max(1, num_epochs // 10)

    for epoch in range(num_epochs):
        te = time.time()
        tr_loss, tr_acc = run_epoch(model, optimizer, dataset,
                                    batch_size, "train", augment)
        vl_loss, vl_acc = run_epoch(model, optimizer, dataset,
                                    batch_size, "val", False)
        et = time.time() - te

        history["train_loss"].append(tr_loss)
        history["train_acc"].append(tr_acc)
        history["val_loss"].append(vl_loss)
        history["val_acc"].append(vl_acc)
        history["epoch_times"].append(et)

        if verbose and ((epoch+1) % log_every == 0 or epoch == 0):
            print(f"  Época {epoch+1:3d}/{num_epochs} | "
                  f"Train  loss={tr_loss:.4f}  acc={tr_acc:.4f} | "
                  f"Val  loss={vl_loss:.4f}  acc={vl_acc:.4f} | "
                  f"{et:.1f}s")

    total_time = time.time() - t0

    # Evaluación en test
    ts_loss, ts_acc = run_epoch(model, optimizer, dataset, batch_size, "test")
    throughput = len(dataset.train_paths) * num_epochs / total_time

    results = {
        **config,
        "history":        history,
        "total_time":     total_time,
        "avg_epoch_time": np.mean(history["epoch_times"]),
        "test_loss":      ts_loss,
        "test_acc":       ts_acc,
        "final_val_acc":  history["val_acc"][-1],
        "best_val_acc":   max(history["val_acc"]),
        "throughput":     throughput,
    }

    if verbose:
        print(f"\n  ✓ Tiempo total: {total_time:.1f}s | "
              f"Avg/época: {np.mean(history['epoch_times']):.2f}s")
        print(f"  ✓ Test → loss={ts_loss:.4f}  acc={ts_acc:.4f}")
        print(f"  ✓ Throughput: {throughput:.0f} imgs/s")

    return results

print("✓ Funciones de entrenamiento definidas")

In [ ]:
# ── 5.4 Funciones de Visualización ──────────────────────────────────────────

def plot_training_curves(history: dict, title: str, save_path: str = None):
    """Curvas de pérdida y exactitud."""
    ep = range(1, len(history["train_loss"])+1)
    fig, axes = plt.subplots(1, 2, figsize=(14, 5))

    axes[0].plot(ep, history["train_loss"], "b-o", lw=2, ms=3, label="Entrenamiento")
    axes[0].plot(ep, history["val_loss"],   "r--s",lw=2, ms=3, label="Validación")
    axes[0].set_title("Curva de Pérdida (Loss)", fontweight="bold")
    axes[0].set_xlabel("Época"); axes[0].set_ylabel("Cross-Entropy Loss")
    axes[0].legend(); axes[0].grid(True, alpha=.3)

    axes[1].plot(ep, [a*100 for a in history["train_acc"]], "b-o", lw=2, ms=3, label="Entrenamiento")
    axes[1].plot(ep, [a*100 for a in history["val_acc"]],   "r--s",lw=2, ms=3, label="Validación")
    axes[1].set_title("Curva de Exactitud (Accuracy)", fontweight="bold")
    axes[1].set_xlabel("Época"); axes[1].set_ylabel("Exactitud (%)")
    axes[1].legend(); axes[1].grid(True, alpha=.3)
    axes[1].set_ylim(0, 100)

    plt.suptitle(title, fontsize=13, fontweight="bold")
    plt.tight_layout()
    if save_path: plt.savefig(save_path, dpi=150, bbox_inches="tight")
    plt.show()


def plot_exp_comparison(results_list, x_key, x_label, title, save_path=None):
    """Tabla de barras comparativa de resultados de experimentos."""
    labels  = [str(r[x_key]) for r in results_list]
    cols    = sns.color_palette("husl", len(results_list))

    metrics = {
        "Exactitud Test (%)":    [r["test_acc"]*100       for r in results_list],
        "Throughput (imgs/s)":   [r["throughput"]          for r in results_list],
        "Tiempo Total (s)":      [r["total_time"]           for r in results_list],
        "Avg Época (s)":         [r["avg_epoch_time"]       for r in results_list],
    }

    fig, axes = plt.subplots(2, 2, figsize=(14, 9))
    for ax, (metric, vals) in zip(axes.flatten(), metrics.items()):
        bars = ax.bar(labels, vals, color=cols, edgecolor="black", linewidth=.5)
        ax.set_title(metric, fontweight="bold")
        ax.set_xlabel(x_label)
        ax.grid(True, alpha=.3, axis="y")
        for bar, v in zip(bars, vals):
            ax.text(bar.get_x()+bar.get_width()/2, bar.get_height()*1.01,
                    f"{v:.1f}", ha="center", va="bottom", fontsize=9)

    plt.suptitle(title, fontsize=13, fontweight="bold")
    plt.tight_layout()
    if save_path: plt.savefig(save_path, dpi=150, bbox_inches="tight")
    plt.show()


def make_table(results_list, config_keys, rename_map=None):
    """DataFrame comparativo de resultados."""
    rows = []
    for r in results_list:
        row = {k: r.get(k) for k in config_keys}
        row["Test Acc (%)"]   = f"{r['test_acc']*100:.2f}"
        row["Val Acc (%)"]    = f"{r['best_val_acc']*100:.2f}"
        row["Total (s)"]      = f"{r['total_time']:.1f}"
        row["Avg/época (s)"]  = f"{r['avg_epoch_time']:.2f}"
        row["Throughput"]     = f"{r['throughput']:.0f}"
        rows.append(row)
    df = pd.DataFrame(rows)
    if rename_map: df = df.rename(columns=rename_map)
    return df

print("✓ Funciones de visualización definidas")

## Sección 6: Entrenamiento Base

Configuración por defecto para establecer la **línea base** (*baseline*).

| Hiperparámetro | Valor |
|---|---|
| Batch Size | 64 |
| Learning Rate | 1 × 10⁻³ |
| Filtros | 64 |
| Dense | 256 |
| Épocas | 30 |

In [ ]:
# ── 6.1 Entrenamiento base ───────────────────────────────────────────────────
BASE_CONFIG = {
    "num_epochs":    30,
    "batch_size":    64,
    "learning_rate": 1e-3,
    "num_filters":   64,
    "dense_size":    256,
    "augment":       True,
}

print("Iniciando entrenamiento base...")
base_res = train_and_evaluate(BASE_CONFIG, dataset, verbose=True)

In [ ]:
# ── 6.2 Curvas de entrenamiento base ────────────────────────────────────────
plot_training_curves(
    base_res["history"],
    title="Entrenamiento Base (lr=1e-3, batch=64, filtros=64, dense=256)",
    save_path="curvas_base.png",
)
print(f"\n  Exactitud Test: {base_res['test_acc']*100:.2f}%  |  "
      f"Mejor Val Acc: {base_res['best_val_acc']*100:.2f}%")

## Sección 7: Experimento 6.1 — Impacto del Tamaño de Lote

Se analizan **batch sizes = {32, 64, 128}** con el resto de hiperparámetros fijos.

| Hipótesis | |
|---|---|
| BS pequeño | Más ruido en gradiente, posible mejor generalización, menor throughput |
| BS grande | Gradiente más estable, mayor throughput, puede requerir ajuste de lr |

In [ ]:
# ── 7.1 Experimentos — Tamaño de Lote ───────────────────────────────────────
BATCH_SIZES  = [32, 64, 128]
BATCH_EPOCHS = 20

batch_results = []

for bs in BATCH_SIZES:
    cfg = {**BASE_CONFIG, "batch_size": bs, "num_epochs": BATCH_EPOCHS}
    res = train_and_evaluate(cfg, dataset, verbose=True)
    batch_results.append(res)
    gc.collect()

print("\n✓ Experimentos de batch size completados")

In [ ]:
# ── 7.2 Resultados — Tamaño de Lote ─────────────────────────────────────────
df_batch = make_table(batch_results, ["batch_size"],
                      rename_map={"batch_size": "Batch Size"})
print("\n📊 TABLA — IMPACTO DEL TAMAÑO DE LOTE")
print(df_batch.to_string(index=False))

plot_exp_comparison(batch_results, "batch_size", "Batch Size",
                    "Experimento 6.1: Impacto del Tamaño de Lote",
                    save_path="exp_batch_size.png")

# Curvas val comparativas
fig, axes = plt.subplots(1, 2, figsize=(14, 5))
cols = ["blue","green","red"]
ep   = range(1, BATCH_EPOCHS+1)
for i, (res, bs) in enumerate(zip(batch_results, BATCH_SIZES)):
    axes[0].plot(ep, res["history"]["val_loss"],
                 color=cols[i], label=f"BS={bs}", lw=2)
    axes[1].plot(ep, [a*100 for a in res["history"]["val_acc"]],
                 color=cols[i], label=f"BS={bs}", lw=2)
for ax, yl, t in zip(axes,
                     ["Pérdida","Exactitud (%)"],
                     ["Val Loss","Val Accuracy"]):
    ax.set_xlabel("Época"); ax.set_ylabel(yl); ax.set_title(t, fontweight="bold")
    ax.legend(); ax.grid(True, alpha=.3)
plt.suptitle("Comparación por Batch Size — Validación", fontweight="bold")
plt.tight_layout()
plt.savefig("exp_batch_curves.png", dpi=150, bbox_inches="tight")
plt.show()

## Sección 8: Experimento 6.2 — Impacto de la Tasa de Aprendizaje

Se evalúan **lr ∈ {10⁻², 10⁻³, 10⁻⁴}** con el resto de hiperparámetros fijos.

In [ ]:
# ── 8.1 Experimentos — Tasa de Aprendizaje ──────────────────────────────────
LEARNING_RATES = [1e-2, 1e-3, 1e-4]
LR_EPOCHS      = 20

lr_results = []

for lr in LEARNING_RATES:
    cfg = {**BASE_CONFIG, "learning_rate": lr, "num_epochs": LR_EPOCHS}
    res = train_and_evaluate(cfg, dataset, verbose=True)
    lr_results.append(res)
    gc.collect()

print("\n✓ Experimentos de learning rate completados")

In [ ]:
# ── 8.2 Resultados — Tasa de Aprendizaje ────────────────────────────────────
df_lr = make_table(lr_results, ["learning_rate"],
                   rename_map={"learning_rate": "Learning Rate"})
print("\n📊 TABLA — IMPACTO DE LA TASA DE APRENDIZAJE")
print(df_lr.to_string(index=False))

plot_exp_comparison(lr_results, "learning_rate", "Learning Rate",
                    "Experimento 6.2: Impacto de la Tasa de Aprendizaje",
                    save_path="exp_lr.png")

# Curvas val comparativas
fig, axes = plt.subplots(1, 2, figsize=(14,5))
lr_labels = ["lr=1e-2","lr=1e-3","lr=1e-4"]
cols = ["blue","green","red"]
ep   = range(1, LR_EPOCHS+1)
for i, (res, lab) in enumerate(zip(lr_results, lr_labels)):
    axes[0].plot(ep, res["history"]["val_loss"],
                 color=cols[i], label=lab, lw=2)
    axes[1].plot(ep, [a*100 for a in res["history"]["val_acc"]],
                 color=cols[i], label=lab, lw=2)
for ax, yl, t in zip(axes,
                     ["Pérdida","Exactitud (%)"],
                     ["Val Loss","Val Accuracy"]):
    ax.set_xlabel("Época"); ax.set_ylabel(yl); ax.set_title(t, fontweight="bold")
    ax.legend(); ax.grid(True, alpha=.3)
plt.suptitle("Comparación por Learning Rate — Validación", fontweight="bold")
plt.tight_layout()
plt.savefig("exp_lr_curves.png", dpi=150, bbox_inches="tight")
plt.show()

## Sección 9: Experimento 6.3 — Impacto del Tamaño de la Red

**9a.** Filtros base: {32, 64, 128}  
**9b.** Neuronas dense: {128, 256, 512}

In [ ]:
# ── 9.1 Experimento 9a — Filtros convolucionales ────────────────────────────
FILTER_SIZES = [32, 64, 128]
NET_EPOCHS   = 15

filter_results = []
print("=== 9a: Filtros Convolucionales ===")
for nf in FILTER_SIZES:
    cfg = {**BASE_CONFIG, "num_filters": nf, "num_epochs": NET_EPOCHS}
    res = train_and_evaluate(cfg, dataset, verbose=True)
    filter_results.append(res)
    gc.collect()

# ── 9.2 Experimento 9b — Neuronas dense ─────────────────────────────────────
DENSE_SIZES  = [128, 256, 512]
dense_results = []
print("\n=== 9b: Neuronas Dense ===")
for ds in DENSE_SIZES:
    cfg = {**BASE_CONFIG, "dense_size": ds, "num_epochs": NET_EPOCHS}
    res = train_and_evaluate(cfg, dataset, verbose=True)
    dense_results.append(res)
    gc.collect()

print("\n✓ Experimentos de tamaño de red completados")

In [ ]:
# ── 9.3 Resultados — Tamaño de Red ──────────────────────────────────────────
df_filt  = make_table(filter_results, ["num_filters"], {"num_filters":"Filtros"})
df_dense = make_table(dense_results,  ["dense_size"],  {"dense_size":"Neuronas Dense"})
print("📊 TABLA — FILTROS CONVOLUCIONALES")
print(df_filt.to_string(index=False))
print("\n📊 TABLA — NEURONAS CAPA DENSA")
print(df_dense.to_string(index=False))

fig, axes = plt.subplots(2, 2, figsize=(14,10))
pal = sns.color_palette("husl")

def add_bars(ax, labels, vals, title, xlabel):
    cols = sns.color_palette("husl", len(labels))
    bars = ax.bar(labels, vals, color=cols, edgecolor="black", lw=.5)
    ax.set_title(title, fontweight="bold")
    ax.set_xlabel(xlabel); ax.grid(True, alpha=.3, axis="y")
    for b,v in zip(bars,vals):
        ax.text(b.get_x()+b.get_width()/2, b.get_height()*1.01,
                f"{v:.1f}", ha="center", fontsize=9)

add_bars(axes[0,0], [str(f) for f in FILTER_SIZES],
         [r["test_acc"]*100 for r in filter_results],
         "Test Acc vs Filtros", "Filtros base")
add_bars(axes[0,1], [str(f) for f in FILTER_SIZES],
         [r["total_time"] for r in filter_results],
         "Tiempo Total vs Filtros", "Filtros base")
add_bars(axes[1,0], [str(d) for d in DENSE_SIZES],
         [r["test_acc"]*100 for r in dense_results],
         "Test Acc vs Neuronas Dense", "Neuronas Dense")
add_bars(axes[1,1], [str(d) for d in DENSE_SIZES],
         [r["total_time"] for r in dense_results],
         "Tiempo Total vs Neuronas Dense", "Neuronas Dense")

plt.suptitle("Experimento 6.3: Impacto del Tamaño de la Red",
             fontsize=13, fontweight="bold")
plt.tight_layout()
plt.savefig("exp_network_size.png", dpi=150, bbox_inches="tight")
plt.show()

## Sección 10: Optimización de Hiperparámetros

Se aplica **Random Search** sobre el espacio de configuraciones.  
Se evalúan 8 configuraciones y se entrena el mejor modelo con más épocas.

In [ ]:
# ── 10.1 Random Search ──────────────────────────────────────────────────────
random.seed(SEED)

SEARCH_SPACE = {
    "learning_rate": [1e-2, 5e-3, 1e-3, 5e-4, 1e-4],
    "batch_size":    [32, 64, 128],
    "num_filters":   [32, 64, 128],
    "dense_size":    [128, 256, 512],
}
N_TRIALS     = 8
HPO_EPOCHS   = 15

# Generar N_TRIALS configuraciones únicas
tried_keys = set()
configs    = []
while len(configs) < N_TRIALS:
    cfg = {k: random.choice(v) for k, v in SEARCH_SPACE.items()}
    key = tuple(cfg.values())
    if key not in tried_keys:
        tried_keys.add(key)
        configs.append(cfg)

print(f"Ejecutando Random Search — {N_TRIALS} configuraciones\n")
hpo_results = []

for i, cfg in enumerate(configs):
    full_cfg = {**cfg, "num_epochs": HPO_EPOCHS, "augment": True}
    print(f"[{i+1}/{N_TRIALS}] lr={cfg['learning_rate']:.0e}  "
          f"batch={cfg['batch_size']}  filtros={cfg['num_filters']}  "
          f"dense={cfg['dense_size']}")
    res = train_and_evaluate(full_cfg, dataset, verbose=False)
    hpo_results.append(res)
    print(f"       → Val Acc: {res['best_val_acc']*100:.2f}%  "
          f"Test Acc: {res['test_acc']*100:.2f}%  "
          f"Tiempo: {res['total_time']:.0f}s")
    gc.collect()

# Ordenar por mejor val_acc
hpo_sorted = sorted(hpo_results, key=lambda r: r["best_val_acc"], reverse=True)

print(f"\n🏆 TOP 3 CONFIGURACIONES:")
for i, r in enumerate(hpo_sorted[:3]):
    print(f"  {i+1}. lr={r['learning_rate']:.0e}  batch={r['batch_size']}  "
          f"filtros={r['num_filters']}  dense={r['dense_size']}")
    print(f"     Val Acc: {r['best_val_acc']*100:.2f}%  |  "
          f"Test Acc: {r['test_acc']*100:.2f}%")

In [ ]:
# ── 10.2 Tabla y visualización HPO ──────────────────────────────────────────
df_hpo = make_table(hpo_sorted,
                    ["learning_rate","batch_size","num_filters","dense_size"],
                    {"learning_rate":"LR","batch_size":"Batch",
                     "num_filters":"Filtros","dense_size":"Dense"})
print("\n📊 TABLA COMPLETA — RANDOM SEARCH")
print(df_hpo.to_string(index=False))

# Scatter plot: lr vs batch_size coloreado por val_acc
fig, ax = plt.subplots(figsize=(9,5))
sc = ax.scatter(
    [np.log10(r["learning_rate"]) for r in hpo_results],
    [r["batch_size"]               for r in hpo_results],
    c=[r["best_val_acc"]*100       for r in hpo_results],
    s=250, cmap="RdYlGn", edgecolors="black", linewidth=.5,
    vmin=min(r["best_val_acc"]*100 for r in hpo_results),
    vmax=max(r["best_val_acc"]*100 for r in hpo_results),
)
plt.colorbar(sc, ax=ax, label="Val Acc (%)")
ax.set_xlabel("log₁₀(Learning Rate)")
ax.set_ylabel("Batch Size")
ax.set_title("Random Search — Exactitud por Configuración", fontweight="bold")
ax.set_xticks([-4,-3,-2]); ax.set_xticklabels(["1e-4","1e-3","1e-2"])
plt.tight_layout()
plt.savefig("hpo_scatter.png", dpi=150, bbox_inches="tight")
plt.show()

# Guardar mejor config
BEST_CONFIG = {
    **{k: hpo_sorted[0][k]
       for k in ["learning_rate","batch_size","num_filters","dense_size"]},
    "num_epochs": 40,
    "augment": True,
}
print(f"\n✓ Mejor configuración seleccionada:")
for k,v in BEST_CONFIG.items():
    print(f"  {k}: {v}")

In [ ]:
# ── 10.3 Entrenamiento del mejor modelo ─────────────────────────────────────
print(f"\n🚀 Entrenando mejor modelo — {BEST_CONFIG['num_epochs']} épocas")
best_res = train_and_evaluate(BEST_CONFIG, dataset, verbose=True)

plot_training_curves(
    best_res["history"],
    title=(f"Mejor Modelo: lr={BEST_CONFIG['learning_rate']:.0e}  "
           f"batch={BEST_CONFIG['batch_size']}  "
           f"filtros={BEST_CONFIG['num_filters']}  "
           f"dense={BEST_CONFIG['dense_size']}"),
    save_path="curvas_mejor_modelo.png",
)

print(f"\n  ✓ Mejor Test Acc: {best_res['test_acc']*100:.2f}%")
print(f"  ✓ Mejor Val Acc : {best_res['best_val_acc']*100:.2f}%")

## Sección 11: Análisis de Precisión Numérica

| Formato | Bits totales | Exponente | Mantisa | Rango máx |
|---|---|---|---|---|
| float32  | 32 | 8 | 23 | ±3.4 × 10³⁸ |
| float16  | 16 | 5 | 10 | ±65 504 |
| bfloat16 | 16 | 8 | 7  | ±3.4 × 10³⁸ |

> **bfloat16** mantiene el mismo rango que float32 pero con menor precisión, siendo el formato nativo de las TPUs de Google.

In [ ]:
# ── 11.1 Experimentos de Precisión Numérica ─────────────────────────────────
PREC_EPOCHS = 20
PREC_BS     = 64
PREC_LR     = 1e-3

DTYPES = {
    "float32":  jnp.float32,
    "float16":  jnp.float16,
    "bfloat16": jnp.bfloat16,
}

precision_results = {}

for dtype_name, dtype in DTYPES.items():
    print(f"\n{'─'*55}")
    print(f"  Precisión: {dtype_name}")

    model     = CNN(num_classes=NUM_CLASSES, num_filters=64, dense_size=256,
                    rngs=nnx.Rngs(SEED))
    optimizer = nnx.Optimizer(model, optax.adam(PREC_LR))

    hist = {"train_loss":[],"train_acc":[],
            "val_loss":[],"val_acc":[],"epoch_times":[]}
    nan_found = False
    t0 = time.time()

    for epoch in range(PREC_EPOCHS):
        te = time.time()
        tr_ls, tr_ac = [], []

        # ── Entrenamiento ─────────────────────────────────────────────
        for batch in dataset.get_batches("train", PREC_BS, shuffle=True):
            # Castear imágenes a la precisión deseada
            imgs = jnp.array(batch["image"], dtype=dtype)
            lbs  = jnp.array(batch["label"])

            # Castear de vuelta a float32 para el modelo (precision segura)
            imgs_f32 = imgs.astype(jnp.float32)

            def loss_fn(model):
                logits = model(imgs_f32, use_running_average=False)
                loss   = optax.softmax_cross_entropy_with_integer_labels(
                             logits, lbs).mean()
                return loss, logits

            gfn = nnx.value_and_grad(loss_fn, has_aux=True)
            (loss, logits), grads = gfn(model)
            optimizer.update(grads)

            if jnp.isnan(loss):
                nan_found = True
                break

            tr_ls.append(float(loss))
            tr_ac.append(float(jnp.mean(jnp.argmax(logits,-1) == lbs)))

        if nan_found:
            print(f"  ⚠ NaN detectado en época {epoch+1} — interrumpiendo")
            break

        # ── Validación ────────────────────────────────────────────────
        vl_ls, vl_ac = [], []
        for batch in dataset.get_batches("val", PREC_BS, shuffle=False):
            imgs_f32 = jnp.array(batch["image"], dtype=dtype).astype(jnp.float32)
            lbs      = jnp.array(batch["label"])
            logits   = model(imgs_f32, use_running_average=True)
            loss     = optax.softmax_cross_entropy_with_integer_labels(
                           logits, lbs).mean()
            vl_ls.append(float(loss))
            vl_ac.append(float(jnp.mean(jnp.argmax(logits,-1) == lbs)))

        et = time.time() - te
        hist["train_loss"].append(np.mean(tr_ls) if tr_ls else float("nan"))
        hist["train_acc"].append(np.mean(tr_ac)  if tr_ac else 0.)
        hist["val_loss"].append(np.mean(vl_ls))
        hist["val_acc"].append(np.mean(vl_ac))
        hist["epoch_times"].append(et)

    total_time = time.time() - t0

    # ── Test ──────────────────────────────────────────────────────────
    ts_ls, ts_ac = [], []
    for batch in dataset.get_batches("test", PREC_BS, shuffle=False):
        imgs_f32 = jnp.array(batch["image"], dtype=dtype).astype(jnp.float32)
        lbs      = jnp.array(batch["label"])
        logits   = model(imgs_f32, use_running_average=True)
        loss     = optax.softmax_cross_entropy_with_integer_labels(logits,lbs).mean()
        ts_ls.append(float(loss)); ts_ac.append(float(jnp.mean(jnp.argmax(logits,-1)==lbs)))

    throughput = len(dataset.train_paths) * PREC_EPOCHS / total_time

    precision_results[dtype_name] = {
        "dtype":          dtype_name,
        "history":        hist,
        "total_time":     total_time,
        "avg_epoch_time": np.mean(hist["epoch_times"]) if hist["epoch_times"] else 0,
        "test_loss":      np.mean(ts_ls),
        "test_acc":       np.mean(ts_ac),
        "best_val_acc":   max(hist["val_acc"]) if hist["val_acc"] else 0.,
        "throughput":     throughput,
        "nan_detected":   nan_found,
    }
    status = "⚠ NaN" if nan_found else "✓"
    print(f"  {status}  Test Acc: {np.mean(ts_ac)*100:.2f}%  |  "
          f"Tiempo: {total_time:.1f}s  |  Throughput: {throughput:.0f} imgs/s")
    gc.collect()

print("\n✓ Análisis de precisión numérica completado")

In [ ]:
# ── 11.2 Visualización de Precisión Numérica ────────────────────────────────
DCOLS = {"float32":"blue","float16":"green","bfloat16":"red"}

fig, axes = plt.subplots(2, 2, figsize=(14,10))

# Curvas de validación
for dname, res in precision_results.items():
    ep = range(1, len(res["history"]["val_loss"])+1)
    axes[0,0].plot(ep, res["history"]["val_loss"],
                   color=DCOLS[dname], label=dname, lw=2)
    axes[0,1].plot(ep, [a*100 for a in res["history"]["val_acc"]],
                   color=DCOLS[dname], label=dname, lw=2)

for ax,yl,t in zip([axes[0,0],axes[0,1]],
                   ["Pérdida","Exactitud (%)"],["Val Loss","Val Accuracy"]):
    ax.set_xlabel("Época"); ax.set_ylabel(yl)
    ax.set_title(t, fontweight="bold"); ax.legend(); ax.grid(True,alpha=.3)

# Barras comparativas
dnames = list(precision_results.keys())
bcols  = [DCOLS[d] for d in dnames]

bars0 = axes[1,0].bar(dnames, [precision_results[d]["test_acc"]*100 for d in dnames],
                      color=bcols, edgecolor="black", alpha=.85)
axes[1,0].set_title("Test Accuracy", fontweight="bold")
axes[1,0].set_ylabel("Exactitud (%)"); axes[1,0].grid(True,alpha=.3,axis="y")
for b in bars0:
    axes[1,0].text(b.get_x()+b.get_width()/2, b.get_height()+.3,
                   f"{b.get_height():.1f}%", ha="center", fontsize=10)

bars1 = axes[1,1].bar(dnames, [precision_results[d]["throughput"] for d in dnames],
                      color=bcols, edgecolor="black", alpha=.85)
axes[1,1].set_title("Throughput (imgs/s)", fontweight="bold")
axes[1,1].set_ylabel("Imágenes/segundo"); axes[1,1].grid(True,alpha=.3,axis="y")
for b in bars1:
    axes[1,1].text(b.get_x()+b.get_width()/2, b.get_height()+1,
                   f"{b.get_height():.0f}", ha="center", fontsize=10)

plt.suptitle("Análisis de Precisión Numérica: float32 vs float16 vs bfloat16",
             fontsize=13, fontweight="bold")
plt.tight_layout()
plt.savefig("precision_comparison.png", dpi=150, bbox_inches="tight")
plt.show()

# Tabla resumen
print("\n📊 TABLA COMPARATIVA — PRECISIÓN NUMÉRICA")
print(f"{'Dtype':<12} {'Test Acc':>10} {'Val Acc':>10} {'Tiempo':>10} "
      f"{'Avg/época':>10} {'Throughput':>12} {'NaN':>5}")
print("-"*65)
for d, r in precision_results.items():
    print(f"{d:<12} {r['test_acc']*100:>9.2f}% {r['best_val_acc']*100:>9.2f}% "
          f"{r['total_time']:>9.1f}s {r['avg_epoch_time']:>9.2f}s "
          f"{r['throughput']:>12.0f} {'Sí⚠' if r['nan_detected'] else 'No✓':>5}")

## Sección 12: Análisis y Discusión

> **Nota**: Complete los valores marcados con `[→ completar]` con los resultados obtenidos en sus experimentos.

---

### 1. ¿Qué ventajas ofrece Flax NNX respecto a implementar modelos directamente en JAX?

Flax NNX introduce varias ventajas fundamentales:

- **Modularidad OOP**: Permite definir modelos como clases Python (`nnx.Module`), facilitando la composición y reutilización de componentes (capas convolucionales, normalización, etc.).
- **Gestión automática del estado**: En JAX puro se deben manejar manualmente los pytrees de parámetros; NNX encapsula `params`, `batch_stats`, y otros estados, simplificando el código.
- **Integración nativa con Optax**: `nnx.Optimizer` vincula el modelo y el optimizador, eliminando boilerplate de actualización de parámetros.
- **API compatible con JIT**: `@nnx.jit` y `nnx.value_and_grad` entienden la estructura del módulo directamente.
- **BatchNorm y Dropout ergonómicos**: El estado mutable (medias/varianzas corrientes, máscara de dropout) es administrado automáticamente.

---

### 2. ¿Qué beneficios aporta la compilación mediante `jax.jit`?

`@nnx.jit` aplica compilación XLA (*Accelerated Linear Algebra*):

- **Reducción de tiempo de ejecución**: Tras la primera compilación, los pasos siguientes son significativamente más rápidos.
- **Fusión de operaciones**: XLA combina kernels (e.g., Conv+BN+ReLU) para minimizar transferencias de datos entre CPU/GPU.
- **Optimización de memoria**: Reutilización de buffers y operaciones in-place.
- **Portabilidad hardware**: El mismo código se ejecuta eficientemente en CPU, GPU y TPU sin cambios manuales.
- **Eliminación de código muerto**: XLA puede omitir operaciones cuya salida no es utilizada.

---

### 3. ¿Qué hiperparámetro tuvo mayor impacto sobre la exactitud final?

→ **Completar con resultados del experimento**

*Análisis esperado:* La **tasa de aprendizaje** suele ser el hiperparámetro de mayor impacto. `lr=1e-3` equilibra velocidad de convergencia y estabilidad con Adam. `lr=1e-2` puede causar oscilaciones; `lr=1e-4` converge demasiado lentamente dentro del mismo número de épocas.

---

### 4. ¿Qué hiperparámetro tuvo mayor impacto sobre el tiempo de entrenamiento?

→ **Completar con resultados del experimento**

*Análisis esperado:* El **batch size** y el **número de filtros** tienen mayor influencia. Mayor batch_size aumenta el throughput (mejor paralelismo GPU). Mayor num_filters incrementa el costo computacional por paso.

---

### 5. ¿Cómo afectó el tamaño de lote al rendimiento observado?

→ **Completar con valores de throughput e exactitud de cada BS**

*Análisis esperado:*
- **BS=32**: Gradientes más ruidosos, potencialmente mejor generalización, menor throughput.
- **BS=64**: Balance entre estabilidad y velocidad (baseline).
- **BS=128**: Mayor throughput, estimación del gradiente más estable; puede requerir ajuste del lr (*linear scaling rule*: lr×2 si BS×2).

---

### 6. ¿Qué diferencias encontró entre float32, float16 y bfloat16?

→ **Completar con valores de exactitud, throughput y si se detectó NaN**

*Análisis esperado:*
- **float32**: Mayor estabilidad numérica; referencia de exactitud; mayor consumo de memoria.
- **float16**: Rango limitado (±65 504) → riesgo de underflow/overflow; puede requerir *loss scaling*; mayor throughput en GPU Tensor Cores.
- **bfloat16**: Mismo rango que float32 (8 bits de exponente); menor precisión relativa; más estable que float16 para entrenamiento; formato nativo de TPUs.

---

### 7. ¿Cuál configuración produjo el mejor balance entre rendimiento y exactitud?

→ **Completar con la configuración identificada en la Sección 10 (HPO)**

La configuración óptima identificada mediante Random Search fue:  
`lr=X, batch=Y, filtros=Z, dense=W` — lograron XX.XX% de test accuracy con YY.Y imgs/s de throughput.

---

### 8. ¿Qué limitaciones encontró al utilizar Google Colab?

- **Tiempo de sesión limitado**: ~12 horas para cuentas gratuitas; experimentos extensos deben guardarse en Drive.
- **VRAM limitada**: GPU T4 tiene 16 GB; modelos grandes pueden causar OOM.
- **Almacenamiento efímero**: Los datos se pierden al cerrar la sesión; necesario re-descargar.
- **Throttling por uso intensivo**: Sesiones con alto uso de GPU pueden verse limitadas o desconectadas.
- **Sin soporte nativo para múltiples GPUs**: En Colab gratuito solo hay 1 GPU/TPU disponible.
- **Latencia de red**: Descarga de datasets grandes puede ser lenta dependiendo de la región.

---

### 9. ¿Qué mejoras futuras propondría para aumentar el desempeño del modelo?

1. **Transfer Learning**: Usar modelos pre-entrenados (EfficientNet, ResNet, ViT) con Flax NNX como extractor de características o fine-tuning.
2. **Augmentation avanzada**: RandAugment, Mixup, CutMix, AutoAugment.
3. **Learning Rate Scheduling**: Cosine Annealing, OneCycleLR, Warmup lineal.
4. **Regularización adicional**: Weight Decay (AdamW), Label Smoothing, Stochastic Depth.
5. **Arquitecturas más profundas**: ResNet-style skip connections para redes más profundas sin degradación.
6. **Self-supervised pre-training**: SimCLR o MAE sobre datos adicionales de maravillas.
7. **Ensemble**: Promediar predicciones de múltiples modelos con diferentes inicializaciones.
8. **Paralelismo multi-dispositivo**: `jax.pmap` o Named Sharding para aprovechar múltiples GPUs/TPUs.

## Sección 13: Resumen Final y Conclusiones

In [ ]:
# ── 13.1 Resumen ejecutivo de todos los experimentos ────────────────────────
print("=" * 65)
print("  RESUMEN FINAL DE EXPERIMENTOS")
print("=" * 65)

print("\n📌 ENTRENAMIENTO BASE:")
print(f"   Test Acc : {base_res['test_acc']*100:.2f}%  |  "
      f"Tiempo: {base_res['total_time']:.1f}s")

print("\n📌 EXP 6.1 — BATCH SIZE:")
for r in batch_results:
    print(f"   BS={r['batch_size']:3d} → Test Acc={r['test_acc']*100:.2f}%  "
          f"Throughput={r['throughput']:.0f} imgs/s  "
          f"Tiempo={r['total_time']:.0f}s")

print("\n📌 EXP 6.2 — LEARNING RATE:")
for r in lr_results:
    print(f"   lr={r['learning_rate']:.0e} → Test Acc={r['test_acc']*100:.2f}%")

print("\n📌 EXP 6.3a — FILTROS:")
for r in filter_results:
    print(f"   Filtros={r['num_filters']:3d} → Test Acc={r['test_acc']*100:.2f}%  "
          f"Tiempo={r['total_time']:.0f}s")

print("\n📌 EXP 6.3b — NEURONAS DENSE:")
for r in dense_results:
    print(f"   Dense={r['dense_size']:3d}  → Test Acc={r['test_acc']*100:.2f}%")

print("\n📌 HPO — MEJOR CONFIGURACIÓN:")
print(f"   lr={BEST_CONFIG['learning_rate']:.0e}  batch={BEST_CONFIG['batch_size']}  "
      f"filtros={BEST_CONFIG['num_filters']}  dense={BEST_CONFIG['dense_size']}")
print(f"   Test Acc: {best_res['test_acc']*100:.2f}%  |  "
      f"Val Acc:  {best_res['best_val_acc']*100:.2f}%")

print("\n📌 ANÁLISIS DE PRECISIÓN NUMÉRICA:")
for dname, res in precision_results.items():
    nan_tag = " ⚠ NaN" if res["nan_detected"] else ""
    print(f"   {dname:<10} → Test Acc={res['test_acc']*100:.2f}%  "
          f"Throughput={res['throughput']:.0f} imgs/s{nan_tag}")

print("\n" + "=" * 65)
print("  ✅ Notebook completado exitosamente")
print("  Estudiante: Jason Jesús Barrantes Sánchez")
print("  Universidad: LEAD University")
print("  Curso: Aprendizaje Distribuido para Computer Vision")
print("=" * 65)

## Conclusiones

*Completar con los valores reales de los experimentos.*

En esta tarea se implementó un sistema de clasificación de imágenes del dataset *Wonders of the World* utilizando **JAX** y **Flax NNX**, ejecutado sobre GPU en Google Colab.

### Hallazgos principales

1. **Arquitectura CNN con Flax NNX**: La arquitectura de tres bloques convolucionales con Global Average Pooling demostró ser efectiva y modular. Flax NNX simplificó significativamente la gestión del estado del modelo en comparación con JAX puro.

2. **Compilación JIT**: `@nnx.jit` redujo el tiempo de ejecución por paso tras la primera compilación, aprovechando la optimización XLA para GPU/TPU.

3. **Hiperparámetros**: La tasa de aprendizaje mostró el mayor impacto en exactitud; el batch size afectó principalmente el throughput. La configuración óptima fue identificada mediante Random Search.

4. **Precisión numérica**: float32 ofreció la mayor estabilidad; bfloat16 mostró buen balance con menor consumo de memoria; float16 requiere precaución adicional por su rango dinámico limitado.

5. **Google Colab**: Las limitaciones de sesión y VRAM restringen la escala de experimentos, pero el entorno es suficiente para validar conceptos de aprendizaje distribuido.

### Trabajo futuro

El uso de Transfer Learning con modelos pre-entrenados (EfficientNet, ViT) y técnicas avanzadas de augmentation representaría la mejora más significativa. Para escalabilidad, `jax.pmap` con múltiples GPUs/TPUs permitiría entrenar modelos más grandes en menor tiempo.